# A6b · Kimi-K3 (KDA + gated MLA + Attention Residuals + latent MoE) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/A6b_kimi_k3.ipynb)

원본: HF 원격 코드 `moonshotai/Kimi-K3` @ `f831ab6` 의 `modeling_kimi_linear.py` (텍스트 모델 `KimiLinearModel`).
내 구현: `miniarch/models/kimi_k3.py`. A7b(Attention Residuals)도 같은 모델 안에 있어서 함께 검증합니다.

- KDA (Kimi Delta Attention): Gated DeltaNet과 같은 delta rule인데 망각 게이트가 key 채널마다 다릅니다. short conv, 저랭크 게이트, lower bound -5
- gated MLA: latent KV 압축, RoPE 없음(NoPE), 출력 sigmoid 게이트
- Attention Residuals: residual을 더하는 대신 이전 블록 출력들을 softmax로 섞어 다음 층 입력을 만듭니다
- latent MoE: sigmoid 라우터 + bias, routed expert는 작은 latent 차원에서 계산, shared expert, SiTU 활성함수

**fla 커널 대체**: 원본 KDA는 fla Triton 커널(`chunk_kda`)을 부릅니다. T4/fp32에서 안 돌 수 있어 원본 builder가
같은 수식의 순수 PyTorch 판(fla의 `naive_recurrent_kda` 루프 + 커널 안 전처리)으로 바꿔 끼웁니다. 나머지는 원본 코드 그대로입니다.

축소 config: 4층 = [KDA, KDA, KDA, MLA], AttnRes 블록 2층(블록 경계 2번), 시퀀스 100 → 내 구현 chunk(64) 2개 + padding.
첫 층 dense, 나머지 MoE(expert 8, top-2, latent 32). 그룹 라우팅 분기(그룹 2개 중 1개)까지 쓰도록 했고(K3 실제는 그룹 1개),
SiTU β를 작게(1.5 / 2.5) 잡아 tanh 포화가 출력에 드러나게 했습니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. causal 누수 → 5. 단계 전용 자체 점검 → 6. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip -q install -U -r requirements-colab.txt
# Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
!pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
!pip -q install -U 'einops'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("A6b")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('moonshotai/Kimi-K3', 'language_model\\.model\\.layers\\.(0|3)\\.', subfolder='')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## causal 누수

뒤쪽 토큰을 바꿨을 때 앞쪽 출력이 변하면 미래 정보가 새고 있다는 뜻입니다.

In [ ]:
import importlib
V.set_deterministic(0)
mine = importlib.import_module(S.CASE.mine_module).verify_pair(S.ref()[0])["mine"].to(dev).eval()
r = V.causal_leak_test(lambda x: mine(x).last_hidden_state, vocab_size=V.cases.VOCAB, seq_len=48, device=dev, atol=1e-4)
report("내 구현 " + r.name, r.ok, r.max_abs)

## 단계 전용 자체 점검

### KDA recurrent == chunk == fla naive

토큰 하나씩 메모리를 갱신하는 정의 그대로의 구현, 학습용 chunk 구현, 원본 builder가 쓰는 fla naive 루프가 같은 값을 내는지 확인합니다.
망각은 채널마다 다르고 [-5, 0) 범위 전체(아주 강한 망각 포함)를 쓰며, 길이 150 · chunk 32로 경계를 여러 번 넘깁니다.

In [ ]:
from miniarch.models import kimi_k3 as K

V.set_deterministic(0)
B_, H_, T_, Dk_, Dv_ = 2, 3, 150, 16, 8
q_ = K.l2norm(torch.randn(B_, H_, T_, Dk_, device=dev)) * Dk_ ** -0.5
k_ = K.l2norm(torch.randn(B_, H_, T_, Dk_, device=dev))
v_ = torch.randn(B_, H_, T_, Dv_, device=dev)
g_ = -5 * torch.sigmoid(3 * torch.randn(B_, H_, T_, Dk_, device=dev))
b_ = torch.rand(B_, H_, T_, device=dev)
rec = K.kda_recurrent(q_, k_, v_, g_, b_)[0]
r = V.same_function_test("KDA recurrent vs chunk(32)", lambda: rec,
                         lambda: K.kda_chunk(q_, k_, v_, g_, b_, chunk_size=32)[0])
report(r.name, r.ok, r.max_abs)
tr = lambda t: t.transpose(1, 2)
r = V.same_function_test("KDA recurrent vs fla naive (원본 builder)", lambda: rec,
                         lambda: tr(S.naive_kda(tr(q_), tr(k_), tr(v_), tr(g_), tr(b_), scale=1.0)[0]))
report(r.name, r.ok, r.max_abs)

### 모델 전체: chunk 모드 == recurrent 모드

같은 가중치로 `delta_mode="chunk"`와 `"recurrent"`의 최종 출력이 같아야 합니다.

In [ ]:
import dataclasses
from miniarch.models import kimi_k3 as K

V.set_deterministic(0)
ref, inp = S.ref()
m = K.verify_pair(ref)["mine"].to(dev).eval()
m2 = K.KimiK3(dataclasses.replace(m.config, delta_mode="recurrent")).to(dev).eval()
m2.load_state_dict(m.state_dict())
x = inp["input_ids"].to(dev)
with torch.no_grad():
    err = (m(x).last_hidden_state - m2(x).last_hidden_state).abs().max().item()
report("모델 chunk vs recurrent", err < 1e-4, err)

### hook 확인: AttnRes 깊이 방향 확률 · MLA attention map · 라우팅 · KDA 망각

- `*_res_probe`: [B, T, 블록 수+1]. 각 층 입력이 이전 블록들(첫 칸 = embedding)과 현재 부분합을 어떤 비율로 섞는지. 합 = 1.
- `attn_probs`: MLA 층 [B, H, T, T], 행 합 = 1. `route_probe`: 토큰마다 고른 expert. `decay_probe`: 채널별 log 망각.

In [ ]:
names = [n for n, _ in m.named_modules() if n.endswith(("res_probe", "attn_probs", "route_probe", "decay_probe"))]
with V.capture(m, names) as st, torch.no_grad():
    m(x)
for n in names:
    if n not in st:   # 0층 attention 입력은 이전 블록이 없어 섞지 않음
        continue
    t = st[n]
    if n.endswith(("res_probe", "attn_probs")):
        err = (t.sum(-1) - 1).abs().max().item()
        extra = f"평균 확률={t.float().mean((0, 1)).cpu().numpy().round(3)}" if n.endswith("res_probe") else ""
        print(f"{n:<42} {tuple(t.shape)}  합 오차={err:.1e} {extra}")
        report(f"{n} 확률 합", err < 1e-5, err)
    elif n.endswith("route_probe"):
        print(f"{n:<42} expert 사용 횟수={torch.bincount(t.flatten(), minlength=m.config.n_experts).tolist()}")
    else:
        print(f"{n:<42} 한 step 보존율 exp(g) 평균={t.exp().mean():.3f}  최소={t.exp().min():.4f}")

### ablation 스위치 동작

스위치를 하나씩 끈 모델이 정상적으로 forward 되는지 확인합니다 (값 비교가 아니라 shape · 유한값 확인).

In [ ]:
for sw in ["use_attn_res", "use_kda", "use_moe", "use_latent_moe", "use_output_gate"]:
    cfg = dataclasses.replace(m.config, **{sw: False})
    with torch.no_grad():
        out = K.KimiK3(cfg).to(dev).eval()(x).logits
    ok = bool(torch.isfinite(out).all()) and tuple(out.shape) == (x.shape[0], x.shape[1], cfg.vocab_size)
    report(f"ablation {sw}=False forward", ok, 0.0)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/A6b_kimi_k3.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_A6b_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")